In [1]:
CUTOFF = 5

In [2]:
import torch
print("cuda available:", torch.cuda.is_available())
print("num gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

dev = torch.device("cuda")

t1 = torch.randn(3, 3, device=dev)
t2 = torch.randn(3, 3, device=dev)
print(t1 + t2)

cuda available: True
num gpus: 2
NVIDIA GeForce RTX 2080 Ti
tensor([[ 1.2819, -1.0704, -1.2185],
        [-0.2482, -1.5697, -1.4930],
        [-0.1246,  2.9825,  2.2564]], device='cuda:0')


In [3]:
import json
from pymatgen.core import Structure

f = open('matbench_mp_e_form.json')
with f as file:
    mp_form = json.load(file)

f = open('mp_gap.json')
with f as file:
    mp_gap = json.load(file)

In [4]:
mp_form_dict = {(Structure.from_dict(i[0])).to(fmt="poscar"): i for i in mp_form['data']}
mp_gap_dict = {(Structure.from_dict(i[0])).to(fmt="poscar"): i for i in mp_gap['data']}

In [26]:
overlap = 0
mp_form_gap = []
for i in mp_form_dict:
    if i in mp_gap_dict:
        mp_form_gap.append(mp_form_dict[i] + [mp_gap_dict[i][1]])
        overlap += 1
overlap

106113

In [27]:
from pymatgen.core import Structure

def structure_to_graph(structure: Structure):
    atomic_num = []
    c_coords = []

    for site in structure:
        atomic_num.append(site.specie.Z)
        c_coords.append(site.coords)

    return atomic_num, c_coords, structure.lattice._matrix

nodes = []
c_coords = []
lattices = []
e_form = []
e_g = []

# for i in mp_form['data']:
for i in mp_form_gap:
            
    e_form_i = i[1]
    e_g_i = i[2]

    structure_i = Structure.from_dict(i[0])
    atomic_num_i, c_coords_i, lattice_i = structure_to_graph(structure_i)

    if len(atomic_num_i) > 20:
        continue

    nodes.append(atomic_num_i)
    c_coords.append(c_coords_i)
    lattices.append(lattice_i)
    e_form.append(e_form_i)
    e_g.append(e_g_i)
    

In [28]:
MLN = max([len(i) for i in nodes])
MLN

20

In [30]:
len(nodes)

53319

In [32]:
lattices = torch.tensor(lattices, dtype=torch.float32)
e_form = torch.tensor(e_form, dtype=torch.float32)
e_g = torch.tensor(e_g, dtype=torch.float32)

In [33]:
def pbc_distances(lattice, c_coords, atom_mask, cutoff=CUTOFF):

    shifts = torch.tensor([[i, j, k] 
                       for i in [-1, 0, 1]
                       for j in [-1, 0, 1]
                       for k in [-1, 0, 1]], dtype=c_coords.dtype, device=lattice.device)

    f_coords = torch.matmul(c_coords, torch.linalg.inv(lattice))
    f_diff = f_coords[:, :, None, :] - f_coords[:, None, :, :]

    all_f_diff = f_diff[:, :, :, None, :] + shifts[None, None, None, :, :]
    all_c_diff = torch.einsum('xabcz,xzd->xabcd', all_f_diff, lattice)
    
    all_dists = torch.norm(all_c_diff, dim=-1)
    envelope = 0.5 * (torch.cos(all_dists * torch.pi / cutoff) + 1.0)
    envelope = envelope.to(dtype=c_coords.dtype)**1

    mask = (atom_mask[:, :, None] & atom_mask[:, None, :])[..., None]
    mask = mask & (all_dists < cutoff)
    mask = mask & (all_dists > 1e-9)  # check self interaction

    nan_safe_diffs = all_dists + (all_dists < 1e-9).to(all_dists.dtype)
    unit_vectors = all_c_diff/nan_safe_diffs[..., None]
    
    return all_dists, envelope * mask.to(all_dists.dtype), mask.to(all_dists.dtype), unit_vectors

# dd = pbc_distances(lattices[:2], c_coords_padded[:2], (nodes_padded[:2] != 0), CUTOFF)
# dd[0][0][0][0] * dd[1][0][0][0] * dd[2][0][0][0]
# dd[3][0][0][0], 


In [34]:
def gaussian_expansion(dist, dmin=0.0, dmax=CUTOFF, steps=64):

    centers = torch.linspace(dmin, dmax, steps, device=dist.device)
    width = (centers[1] - centers[0]) * 1.0 
    gauss = torch.exp(-((dist[..., None] - centers) ** 2) / (2 * width**2))

    return gauss

# dd = pbc_distances(lattices[:2], c_coords_padded[:2], (nodes_padded[:2] != 0), CUTOFF)
# gg = gaussian_expansion(dd[0], 0.0, CUTOFF, 5) * dd[1][..., None]
# # gg[0][0][0], gg[0][0][-1]
# # gg[0][0][5], dd[0][0][0][5], dd[1][0][0][5]
# [[len([i for i in e if i > 1e-4]) for e in j] for j in (dd[0] * dd[1] * dd[2])[0]]

In [36]:
class CDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.nodes, self.coords, self.lattice, self.r_eg, self.r_ef = data

    def __len__(self):
        return len(self.nodes)

    def __getitem__(self, i):

        return (self.nodes[i], self.coords[i], self.lattice[i], self.r_eg[i], self.r_ef[i])


In [ ]:
class GNNAttention(torch.nn.Module):
    def __init__(self, bond_dim, d_model=1, dropout_rate=0.0):
        super().__init__()
        self.d_model = d_model
        self.dropout_rate = dropout_rate
        
        self.neighbor_ffn = torch.nn.Linear(d_model, d_model, bias=True)
        self.bond_ffn = torch.nn.Linear(d_model * 3, d_model, bias=True)
        self.s_ffn = torch.nn.Linear(d_model * 2, d_model, bias=False)
        self.g_ffn = torch.nn.Linear(d_model * 2, d_model, bias=False)

        self.dropout = torch.nn.Dropout(dropout_rate)
        self.relu1 = torch.nn.LeakyReLU(0.2) 

        self.s = torch.nn.Sigmoid()
        self.g = torch.nn.SiLU()#.SELU()


    def forward(self, inputs):
        atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors = inputs

        B, N, d = atom_features.shape
        K = bond_features.shape[3]

        itself = atom_features[:, :, None, None, :].expand(B, N, N, K, d)
        neighbors = atom_features[:, None, :, None, :].expand(B, N, N, K, d)  # batch_size, M_L_N, M_L_N, 125, d_model

        bond_features = torch.cat([itself, neighbors, bond_features], dim=-1)
        bond_features = self.g(self.bond_ffn(bond_features)) * bond_mask_envelope

        neighbors = self.g(self.neighbor_ffn(neighbors)) * bond_features * bond_mask_envelope   # batch_size, M_L_N, M_L_N, 125, d_model
        
        vec_messages = unit_vectors[:, :, :, :, None, :] * neighbors[:, :, :, :, :, None]
        vec_messages = torch.sum(vec_messages, dim=-3)  # batch_size, M_L_N, M_L_N, self.d_model, 3
        vec_messages = torch.sum(vec_messages, dim=-3)  # batch_size, M_L_N, self.d_model, 3
        
        neighbors = self.dropout(neighbors)
        neighbors = torch.sum(neighbors, dim=-2)  # batch_size, M_L_N, M_L_N, self.d_model
        neighbors = torch.sum(neighbors, dim=-2)  # batch_size, M_L_N, self.d_model

        return neighbors, bond_features, vec_messages

att = GNNAttention(bond_dim=4, d_model=4)

# node_i = nodes_padded[:2, ..., None].to(dtype=torch.float32)
# dist_i, em, msk, uv = pbc_distances(
#     lattices[:2], 
#     c_coords_padded[:2], 
#     (nodes_padded[:2] != 0),
#     CUTOFF)

# dist_i = gaussian_expansion(dist_i, 0.0, CUTOFF, 4) * msk[..., None]
# att([node_i, dist_i, em[..., None]])[0], node_i[0]


In [39]:
class GRU(torch.nn.Module):
    def __init__(self, d_model, dropout_rate=0.0):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.d_model = d_model

        self.z_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.z_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

        self.r_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.r_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

        self.mbar_kernel_1 = torch.nn.Linear(d_model, d_model, bias=True)
        self.mbar_kernel_2 = torch.nn.Linear(d_model, d_model, bias=True)

    def forward(self, inputs):
        h, m = inputs

        mask = h.sum(dim=-1)
        if h.dim() == 4:
            print("ERROR!!! ")

        mask = (mask != 0).to(h.dtype)[..., None]

        z = ( self.z_kernel_1(h) + self.z_kernel_2(m) ) * mask
        z = torch.sigmoid(z) * mask

        r = ( self.r_kernel_1(h) + self.r_kernel_2(m) ) * mask
        r = torch.sigmoid(r) * mask

        mbar = torch.tanh(( self.mbar_kernel_1(m) + self.mbar_kernel_2(r * h) ) * mask)

        out = (1.0 - z) * h + z * mbar
        return out
    
class TemporalAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads=4, dropout_rate=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_K = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_V = torch.nn.Linear(d_model, d_model, bias=False)
        self.W_O = torch.nn.Linear(d_model, d_model, bias=False)
        self.dropout = torch.nn.Dropout(dropout_rate)
        self.norm = torch.nn.LayerNorm(d_model)

    def forward(self, query, history):

        B, N, d = query.shape
        L = history.shape[2]

        Q = self.W_Q(query).unsqueeze(2)  # (B, N, 1, d)
        K = self.W_K(history)  # (B, N, L, d)
        V = self.W_V(history)  # (B, N, L, d)

        Q = Q.view(B, N, 1, self.n_heads, self.d_k).transpose(2, 3)  # (B, N, heads, 1, d_k)
        K = K.view(B, N, L, self.n_heads, self.d_k).transpose(2, 3)  # (B, N, heads, L, d_k)
        V = V.view(B, N, L, self.n_heads, self.d_k).transpose(2, 3)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (B, N, heads, 1, L)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, V)  # (B, N, heads, 1, d_k)
        out = out.transpose(2, 3).contiguous().view(B, N, 1, d)
        out = out.squeeze(2)  # (B, N, d)
        out = self.W_O(out)

        return self.norm(query + out)

In [ ]:
class EdgeNetwork(torch.nn.Module):
    def __init__(self, g_exp_steps, bond_dim, steps=2, d_model=4, dropout_rate=0.0, n_heads=1, lookback=None):
        super().__init__()

        self.d_model = d_model
        self.dropout_rate = dropout_rate
        self.steps = steps

        self.lookback = lookback if lookback is not None else steps

        self.input_atom_ffn = torch.nn.Linear(d_model, d_model, bias=False)
        self.input_bond_ffn = torch.nn.Linear(g_exp_steps, bond_dim, bias=False)


        self.vec_mes_ffn = torch.nn.ModuleList([torch.nn.Linear(d_model, d_model, bias=False) for _ in range(self.steps)])
        
        self.layers = torch.nn.ModuleList([GNNAttention(bond_dim=bond_dim, d_model=self.d_model, dropout_rate=self.dropout_rate) 
                        for _ in range(self.steps)])
        
        self.dense = torch.nn.ModuleList([torch.nn.Linear(d_model, d_model, bias=False) for _ in range(self.steps)])
        self.dropout = torch.nn.ModuleList([torch.nn.Dropout(p=dropout_rate) for _ in range(self.steps)])
        self.prelu = torch.nn.ModuleList([torch.nn.LeakyReLU(0.2) for _ in range(self.steps)])

        self.gru = torch.nn.ModuleList([GRU(self.d_model) for _ in range(self.steps)])
        self.temporal_attn = torch.nn.ModuleList(
            [TemporalAttention(d_model, n_heads=n_heads, dropout_rate=dropout_rate)
             for _ in range(self.steps)])

        self.vec_gate = torch.nn.ModuleList([torch.nn.Linear(self.d_model, self.d_model, bias=True) for _ in range(self.steps)])

    def forward(self, inputs):

        atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors = inputs
        
        atom_features = self.input_atom_ffn(atom_features)
        bond_features = self.input_bond_ffn(bond_features) * bond_mask_envelope

        mask = (atom_features.abs().sum(-1) > 1e-12).to(dtype=atom_features.dtype)[..., None]

        vec_mes = torch.zeros(atom_features.shape[0], atom_features.shape[1], atom_features.shape[2], 3, 
                       device=atom_features.device, dtype=atom_features.dtype)
        
        history = [atom_features]
   
        for step in range(self.steps):

            step = 0 # use for weight recycling

            # message passing
            neighbors, bond_features, vec_messages = self.layers[step]([atom_features, bond_features, bond_mask_envelope, main_mask, unit_vectors])

            # update
            x_star = atom_features + neighbors
            l_t = min(self.lookback, len(history))
            H = torch.stack(history[-l_t:], dim=2)  # (B, N, l_t, d_model)
            atom_features = self.temporal_attn[step](x_star, H)
            history.append(atom_features)

            atom_features = self.gru[step]([atom_features, neighbors])
            # atom_features = atom_features + neighbors

            vec_mes = vec_mes + vec_messages
            vec_mes = self.vec_mes_ffn[step](vec_mes.transpose(-1, -2)).transpose(-1, -2)

            vec_norm = torch.sqrt(torch.sum(vec_mes ** 2, dim=-1) + 1e-8) * mask
            # atom_features = atom_features + vec_norm
            atom_features = atom_features + torch.sigmoid(self.vec_gate[step](atom_features)) * vec_norm

            atom_features = self.dense[step](atom_features)
            atom_features = self.dropout[step](atom_features)
            atom_features = self.prelu[step](atom_features)

        return atom_features
    
mp = EdgeNetwork(steps=4, g_exp_steps=4, bond_dim=4, d_model=1)

# node_i = nodes_padded[:2, ..., None].to(dtype=torch.float32)
# dist_i, me, msk, uv = pbc_distances(
#     lattices[:2], 
#     c_coords_padded[:2], 
#     (nodes_padded[:2] != 0),
#     4)
# dist_i = gaussian_expansion(dist_i, 0.0, CUTOFF, 4)

# mp([node_i, dist_i, me[..., None], msk[..., None], uv])[0], node_i[0]

In [41]:
def custom_l1_loss(prediction, target, mask=None):
    error = torch.abs(prediction - target) #** 2
    if mask is not None:
        return torch.sum(error * mask) / (torch.sum(mask)*3)
    else:
        return torch.mean(error)

def train(model, optimizer, inputs, targets, training, ep=40, batch_i=None, batches=None):

    nodes, coords, lattice = inputs
    r_eg, r_ef = targets

    nodes   = nodes.to(dev, non_blocking=True)
    coords  = coords.to(dev, non_blocking=True)
    lattice = lattice.to(dev, non_blocking=True)

    r_eg  = r_eg.to(dev, non_blocking=True)
    r_ef  = r_ef.to(dev, non_blocking=True)

    num_atoms = (nodes != 0).sum(dim=1).float()
    p_eg, p_ef = model((nodes, coords, lattice))
    
    eg_loss = custom_l1_loss(p_eg[..., 0], r_eg)
    ef_loss = custom_l1_loss(p_ef[..., 0] / num_atoms, r_ef)
    

    if (batch_i is not None) and (batches is not None):
        print(f"\r\tbatch: {batch_i + 1}/{batches + 1} || batch loss = {eg_loss.item():.4g} \t {ef_loss.item():.4g}", end="")

    loss = 0 + ef_loss

    if training:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return eg_loss.item(), ef_loss.item()
    

In [42]:
class Catformer(torch.nn.Module):
    def __init__(self, d_model, dropout_rate=0.1):
        super().__init__()

        self.dropout_rate = dropout_rate
        self.d_model = d_model
        self.g_exp_steps = 256
        self.bond_dim = 64

        self.embedding = torch.nn.Embedding(99, self.d_model)
        self.b_w = torch.nn.Linear(self.d_model, self.d_model, bias=False)

        self.mp = EdgeNetwork(steps=4, g_exp_steps=self.g_exp_steps, bond_dim=self.bond_dim, d_model=self.d_model, dropout_rate=self.dropout_rate, n_heads=1, lookback=None)

        self.mid1 = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.relu1 = torch.nn.SiLU()#.LeakyReLU(0.2) 
        self.dropout1 = torch.nn.Dropout(self.dropout_rate)

        self.a1_ef = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.a2_ef = torch.nn.Linear(self.d_model, self.d_model, bias=False)
        self.a3_ef = torch.nn.Linear(self.d_model, 1, bias=True)

        self.a1_eg = torch.nn.Linear(self.d_model, self.d_model, bias=True)
        self.a2_eg = torch.nn.Linear(self.d_model, self.d_model, bias=True)
        self.a3_eg = torch.nn.Linear(self.d_model, 1, bias=True)

    def forward(self, inputs):

        node, c_coords, lattice_mat = inputs

        node_mask = (node != 0)
        node = self.embedding(node) 
        node = node * node_mask.type(node.dtype)[..., None]
        node = self.b_w(node)

        dist, dist_mask_envelope, msk, unit_vec = pbc_distances(lattice_mat, c_coords, node_mask, CUTOFF)
        dist = gaussian_expansion(dist, 0.0, CUTOFF, self.g_exp_steps) * msk[..., None]

        x = self.mp([node, dist, dist_mask_envelope[..., None], msk[..., None], unit_vec])  # batch, num_atoms, d_model
        
        x = self.mid1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        node_mask = node_mask.type(node.dtype)[..., None]
        
        eg = x.sum(dim=1) / node_mask.sum(dim=1)  # batch, d_model
        eg = self.a1_eg(eg)  # batch, d_model
        eg = torch.nn.functional.silu(eg)
        eg = self.a2_eg(eg)  # batch, d_model
        eg = torch.nn.functional.silu(eg)
        eg = self.a3_eg(eg)  # batch, 1

        ef = self.a1_ef(x)  # batch, num_atoms, d_model
        ef = torch.nn.functional.silu(ef) * node_mask
        ef = self.a2_ef(ef)  # batch, num_atoms, d_model
        ef = torch.nn.functional.silu(ef) * node_mask
        ef = self.a3_ef(ef)  # batch, num_atoms, 1
        ef = torch.sum(ef, dim=1)  # batch, 1

        return eg, ef

In [43]:
def collate_fn(batch):
    nodes_list, coords_list, lattices, e_g, e_f = zip(*batch)
    max_atoms = max(len(n) for n in nodes_list)
    
    nodes_padded = torch.zeros(len(batch), max_atoms, dtype=torch.int32)
    coords_padded = torch.zeros(len(batch), max_atoms, 3, dtype=torch.float32)
    
    for i, (n, c) in enumerate(zip(nodes_list, coords_list)):
        L = len(n)
        nodes_padded[i, :L] = n
        coords_padded[i, :L] = c

    lattices = torch.stack(lattices)
    e_g = torch.stack(e_g)
    e_f = torch.stack(e_f)

    return nodes_padded, coords_padded, lattices, e_g, e_f

In [44]:
nodes_raw = [torch.tensor(n, dtype=torch.int32) for n in nodes]
coords_raw = [torch.tensor(c, dtype=torch.float32) for c in c_coords]

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=18012019)

fold_results_eg = []
fold_results_ef = []

for fold, (train_idx, test_idx) in enumerate(kf.split(nodes_raw)):

    rng = torch.Generator().manual_seed(42)
    perm = torch.randperm(len(train_idx), generator=rng)
    train_idx = train_idx[perm]
    
    val_size = len(train_idx) // 9
    val_idx = train_idx[:val_size]
    train_idx = train_idx[val_size:]

    training_set = torch.utils.data.DataLoader(
        CDataset(([nodes_raw[i] for i in train_idx],
                  [coords_raw[i] for i in train_idx],
                  lattices[train_idx],
                  e_g[train_idx],
                  e_form[train_idx])), batch_size=8, shuffle=True, drop_last=True, collate_fn=collate_fn)
    
    valid_set = torch.utils.data.DataLoader(
        CDataset(([nodes_raw[i] for i in val_idx],
                  [coords_raw[i] for i in val_idx],
                  lattices[val_idx],
                  e_g[val_idx],
                  e_form[val_idx])), batch_size=8, shuffle=False, drop_last=False, collate_fn=collate_fn)
    
    test_set = torch.utils.data.DataLoader(
        CDataset(([nodes_raw[i] for i in test_idx],
                  [coords_raw[i] for i in test_idx],
                  lattices[test_idx],
                  e_g[test_idx],
                  e_form[test_idx])), batch_size=8, shuffle=False, drop_last=False, collate_fn=collate_fn)

    model = Catformer(64).to(dev)
    model = torch.nn.DataParallel(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for t in range(30):

        ave_loss_eg, ave_loss_ef = 0, 0
        ave_valid_loss_eg, ave_valid_loss_ef = 0, 0

        model.train()

        for batch, (n, c, la, eg, ef) in enumerate(training_set):
            egl, efl = train(model, optimizer, (n, c, la), (eg, ef), True, ep=t, batch_i=batch, batches=len(training_set))
            ave_loss_eg += egl
            ave_loss_ef += efl

        model.eval()
        for batch, (n, c, la, eg, ef) in enumerate(valid_set):
            egl, efl = train(model, optimizer, (n, c, la), (eg, ef), False, ep=t)
            ave_valid_loss_eg += egl
            ave_valid_loss_ef += efl

        print(f"\repoch: {t + 1} \t | \t eg_loss = {(ave_loss_eg/len(training_set)):.4g} \t ef_loss =  {(ave_loss_ef/len(training_set)):.4g} \t | \t val eg_loss = {(ave_valid_loss_eg/len(valid_set)):.4g} \t val ef_loss = {(ave_valid_loss_ef/len(valid_set)):.4g}")

    model.eval()
    test_mae_eg = 0
    test_mae_ef = 0
    count = 0
  
    with torch.no_grad():
        for n, c, la, eg, ef in test_set:
            n, c, la, eg, ef = n.to(dev), c.to(dev), la.to(dev), eg.to(dev), ef.to(dev)
            pred_eg, pred_ef = model((n, c, la))
            num_atoms = (n != 0).sum(dim=1).float()
            test_mae_eg += torch.abs(pred_eg[..., 0] - eg).sum().item()
            test_mae_ef += torch.abs(pred_ef[..., 0] / num_atoms - ef).sum().item()
            count += len(eg)
    
    fold_mae_ef = test_mae_ef / count
    fold_mae_eg = test_mae_eg / count

    fold_results_eg.append(fold_mae_eg)
    fold_results_ef.append(fold_mae_ef)
    print(f"Fold {fold + 1} \t Eg MAE = {fold_mae_eg:.4f} eV \t Ef MAE = {fold_mae_ef:.4f} eV/atom")

print(f"Mean Eg MAE: {sum(fold_results_eg)/5:.4f} ± {torch.tensor(fold_results_eg).std().item():.4f}")
print(f"Mean Ef MAE: {sum(fold_results_ef)/5:.4f} ± {torch.tensor(fold_results_ef).std().item():.4f}")